# Análisis Exploratorio de Datos Textuales (EDA Texto)
## TFG — Magnificent 7: Sentimiento y Riesgo de Mercado

**Fuentes:** GDELT (noticias, 2019–2026) + Reddit (posts, 2019–2026)  
**Empresas:** AAPL, AMZN, GOOGL, META, MSFT, NVDA, TSLA  
**Autor:** Daniel Palacios García

In [1]:
# ── Imports y configuración global ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from wordcloud import WordCloud, STOPWORDS
import warnings
warnings.filterwarnings('ignore')

# Estilo y paleta
sns.set_style('whitegrid')
TICKERS = ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']
EMPRESAS = {'AAPL': 'Apple', 'AMZN': 'Amazon', 'GOOGL': 'Alphabet',
            'META': 'Meta', 'MSFT': 'Microsoft', 'NVDA': 'NVIDIA', 'TSLA': 'Tesla'}
COLORES = {t: plt.cm.tab10(i) for i, t in enumerate(TICKERS)}

# Rutas
ROOT = Path('..') 
FIG_DIR = ROOT / 'docs' / 'figuras'
FIG_DIR.mkdir(parents=True, exist_ok=True)

def guardar(fig, nombre):
    """Guarda figura en docs/figuras/ con DPI alto."""
    fig.savefig(FIG_DIR / nombre, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  → Guardada: docs/figuras/{nombre}")

print("Configuración cargada correctamente.")

Configuración cargada correctamente.


---
## 1. Carga de datos

In [2]:
# ── 1. Carga de datos ──────────────────────────────────────────────────────
DATA = ROOT / 'data' / 'clean'

# GDELT noticias (diario)
news = pd.read_csv(DATA / 'news_clean_daily.csv', parse_dates=['fecha'])
news = news[news['ticker'].isin(TICKERS)].copy()

# Reddit (diario)
reddit = pd.read_csv(DATA / 'reddit_clean_daily.csv', parse_dates=['fecha'])
reddit = reddit[reddit['ticker'].isin(TICKERS)].copy()

# Reddit posts individuales (para word clouds)
posts = pd.read_csv(DATA / 'reddit_clean_posts.csv', parse_dates=['fecha'],
                     low_memory=False)
posts = posts[posts['ticker'].isin(TICKERS)].copy()

print("=" * 65)
print("GDELT — news_clean_daily.csv")
print(f"  Shape: {news.shape}")
print(f"  Periodo: {news['fecha'].min().date()} → {news['fecha'].max().date()}")
print(f"  Tickers: {sorted(news['ticker'].unique())}")
print(f"  Columnas: {list(news.columns)}")
print()
print("Reddit — reddit_clean_daily.csv")
print(f"  Shape: {reddit.shape}")
print(f"  Periodo: {reddit['fecha'].min().date()} → {reddit['fecha'].max().date()}")
print(f"  Tickers: {sorted(reddit['ticker'].unique())}")
print(f"  Columnas: {list(reddit.columns)}")
print()
print("Reddit posts — reddit_clean_posts.csv")
print(f"  Shape: {posts.shape}")
print(f"  Periodo: {posts['fecha'].min().date()} → {posts['fecha'].max().date()}")
print(f"  Subreddits: {sorted(posts['subreddit'].dropna().unique())}")
print("=" * 65)

GDELT — news_clean_daily.csv
  Shape: (18409, 10)
  Periodo: 2019-01-01 → 2026-03-31
  Tickers: ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']
  Columnas: ['fecha', 'ticker', 'tone_mean', 'tone_std', 'tone_median', 'tone_min', 'tone_max', 'n_noticias', 'pct_negativo', 'pct_positivo']

Reddit — reddit_clean_daily.csv
  Shape: (15047, 10)
  Periodo: 2019-01-01 → 2026-04-02
  Tickers: ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']
  Columnas: ['fecha', 'ticker', 'n_posts', 'score_mean', 'score_std', 'score_median', 'n_comments_mean', 'upvote_ratio_mean', 'pct_posts_wsb', 'pct_con_texto']

Reddit posts — reddit_clean_posts.csv
  Shape: (99911, 12)
  Periodo: 2019-01-01 → 2026-04-02
  Subreddits: ['StockMarket', 'investing', 'options', 'stocks', 'wallstreetbets']


---
## 2. GDELT — Estadísticas descriptivas del tono por empresa

In [3]:
# ── 2. Estadísticas descriptivas del tono GDELT ───────────────────────────
# Tabla resumen por empresa
stats_tono = news.groupby('ticker').agg(
    dias=('fecha', 'count'),
    tono_medio=('tone_mean', 'mean'),
    tono_mediana=('tone_mean', 'median'),
    tono_std=('tone_mean', 'std'),
    tono_min=('tone_min', 'min'),
    tono_max=('tone_max', 'max'),
    noticias_dia_media=('n_noticias', 'mean'),
    noticias_dia_mediana=('n_noticias', 'median'),
    pct_neg_medio=('pct_negativo', 'mean'),
    pct_pos_medio=('pct_positivo', 'mean'),
).round(3)

print("Estadísticas descriptivas del tono GDELT (2019–2026)")
print("=" * 90)
display(stats_tono)

Estadísticas descriptivas del tono GDELT (2019–2026)


,dias,tono_medio,tono_mediana,tono_std,tono_min,tono_max,noticias_dia_media,noticias_dia_mediana,pct_neg_medio,pct_pos_medio
ticker,,,,,,,,,,
AAPL,2630,-0.235,-0.123,0.914,-31.469,31.746,449.119,410.5,48.734,47.911
AMZN,2630,0.604,0.731,0.832,-24.096,29.865,190.923,176.0,34.162,63.249
GOOGL,2629,-0.245,-0.146,1.145,-18.000,26.984,65.056,49.0,46.422,49.796
META,2630,0.065,0.138,0.632,-23.333,29.931,324.814,330.0,44.166,52.152
MSFT,2630,0.343,0.427,0.650,-33.333,35.135,1054.300,1012.0,41.594,54.749
NVDA,2630,0.632,0.705,0.688,-28.571,21.429,309.064,184.0,35.009,60.853
TSLA,2630,-0.124,-0.030,0.769,-22.549,29.167,171.580,148.0,46.843,49.214


In [4]:
# ── 2b. Boxplots del tono medio diario por empresa ────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
data_box = [news.loc[news['ticker'] == t, 'tone_mean'].dropna() for t in TICKERS]
bp = ax.boxplot(data_box, labels=TICKERS, patch_artist=True, showfliers=True,
                flierprops=dict(marker='.', markersize=2, alpha=0.3))
for patch, t in zip(bp['boxes'], TICKERS):
    patch.set_facecolor(COLORES[t])
    patch.set_alpha(0.7)
ax.axhline(0, color='grey', ls='--', lw=0.8)
ax.set_ylabel('Tono medio diario (V2Tone GDELT)')
ax.set_title('Distribución del tono medio diario por empresa — GDELT 2019–2026')
fig.tight_layout()
guardar(fig, 'eda_texto_tono_distribucion.png')

  → Guardada: docs/figuras/eda_texto_tono_distribucion.png


---
## 3. GDELT — Evolución temporal mensual del tono

In [5]:
# ── 3. Evolución temporal mensual del tono (subplots, fill verde/rojo) ─────
news['mes'] = news['fecha'].dt.to_period('M')
tono_mensual = news.groupby(['mes', 'ticker'])['tone_mean'].mean().reset_index()
tono_mensual['mes_dt'] = tono_mensual['mes'].dt.to_timestamp()

fig, axes = plt.subplots(4, 2, figsize=(16, 14), sharex=True, sharey=True)
axes_flat = axes.flatten()

for i, t in enumerate(TICKERS):
    ax = axes_flat[i]
    sub = tono_mensual[tono_mensual['ticker'] == t].sort_values('mes_dt')
    ax.plot(sub['mes_dt'], sub['tone_mean'], color=COLORES[t], lw=1.2)
    ax.fill_between(sub['mes_dt'], sub['tone_mean'], 0,
                    where=sub['tone_mean'] >= 0, color='green', alpha=0.25,
                    interpolate=True)
    ax.fill_between(sub['mes_dt'], sub['tone_mean'], 0,
                    where=sub['tone_mean'] < 0, color='red', alpha=0.25,
                    interpolate=True)
    ax.axhline(0, color='grey', ls='--', lw=0.6)
    ax.set_title(f"{t} — {EMPRESAS[t]}", fontsize=11, fontweight='bold')
    ax.set_ylabel('Tono medio')

# Ocultar subplot sobrante (posición 8)
axes_flat[7].set_visible(False)

fig.suptitle('Evolución mensual del tono GDELT por empresa — 2019–2026',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
guardar(fig, 'eda_texto_tono_temporal.png')

  → Guardada: docs/figuras/eda_texto_tono_temporal.png


---
## 4. GDELT — Volumen de noticias por empresa y año (stacked bar)

In [6]:
# ── 4. Volumen de noticias GDELT por empresa y año (stacked bar) ──────────
news['anio'] = news['fecha'].dt.year
vol_anio = news.groupby(['anio', 'ticker'])['n_noticias'].sum().unstack(fill_value=0)
vol_anio = vol_anio[TICKERS]  # orden consistente

fig, ax = plt.subplots(figsize=(12, 6))
vol_anio.plot(kind='bar', stacked=True, ax=ax,
              color=[COLORES[t] for t in TICKERS], edgecolor='white', lw=0.3)
ax.set_title('Volumen total de noticias GDELT por empresa y año — 2019–2026',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('Número total de noticias')
ax.legend(title='Ticker', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
plt.xticks(rotation=0)
fig.tight_layout()
guardar(fig, 'eda_texto_volumen_noticias.png')

  → Guardada: docs/figuras/eda_texto_volumen_noticias.png


---
## 5. GDELT — Correlación del tono diario entre empresas

In [7]:
# ── 5. Correlación del tono diario entre empresas (heatmap) ───────────────
tono_pivot = news.pivot_table(index='fecha', columns='ticker',
                               values='tone_mean', aggfunc='mean')
tono_pivot = tono_pivot[TICKERS]
corr_tono = tono_pivot.corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
mask = np.triu(np.ones_like(corr_tono, dtype=bool), k=1)
sns.heatmap(corr_tono, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mask, square=True, linewidths=0.5, ax=ax,
            vmin=-0.2, vmax=1.0,
            cbar_kws={'label': 'Correlación de Pearson'})
ax.set_title('Correlación del tono diario GDELT entre empresas — 2019–2026',
             fontsize=12, fontweight='bold')
fig.tight_layout()
guardar(fig, 'eda_texto_correlaciones_heatmap.png')

  → Guardada: docs/figuras/eda_texto_correlaciones_heatmap.png


---
## 6. Reddit — Estadísticas descriptivas

In [8]:
# ── 6. Estadísticas descriptivas Reddit ───────────────────────────────────
stats_reddit = reddit.groupby('ticker').agg(
    dias=('fecha', 'count'),
    posts_dia_medio=('n_posts', 'mean'),
    posts_dia_mediana=('n_posts', 'median'),
    posts_dia_max=('n_posts', 'max'),
    score_medio=('score_mean', 'mean'),
    score_mediana=('score_mean', 'median'),
    comentarios_medio=('n_comments_mean', 'mean'),
    pct_wsb_medio=('pct_posts_wsb', 'mean'),
    pct_con_texto_medio=('pct_con_texto', 'mean'),
).round(2)

print("Estadísticas descriptivas de actividad Reddit (2019–2026)")
print("=" * 90)
display(stats_reddit)

# Total de posts por empresa
total_posts = posts.groupby('ticker').size().reindex(TICKERS)
print(f"\nTotal de posts individuales por empresa:")
for t in TICKERS:
    print(f"  {t}: {total_posts[t]:,}")
print(f"  TOTAL: {total_posts.sum():,}")

Estadísticas descriptivas de actividad Reddit (2019–2026)


,dias,posts_dia_medio,posts_dia_mediana,posts_dia_max,score_medio,score_mediana,comentarios_medio,pct_wsb_medio,pct_con_texto_medio
ticker,,,,,,,,,
AAPL,2357,5.44,4.0,79,105.28,17.50,40.93,42.05,68.72
AMZN,2252,4.15,3.0,48,130.31,17.50,45.00,43.94,68.72
GOOGL,1606,2.24,2.0,21,112.78,9.67,43.36,33.07,76.11
META,1713,5.40,3.0,186,130.15,22.67,47.10,49.77,64.31
MSFT,2269,4.77,3.0,160,114.81,17.00,49.19,36.66,75.98
NVDA,2296,8.72,5.0,239,103.92,26.00,40.05,50.20,66.56
TSLA,2554,13.34,7.0,363,142.98,37.80,44.72,64.64,50.55



Total de posts individuales por empresa:
  AAPL: 12,822
  AMZN: 9,342
  GOOGL: 3,595
  META: 9,251
  MSFT: 10,822
  NVDA: 20,017
  TSLA: 34,062
  TOTAL: 99,911


---
## 7. Reddit — Evolución temporal mensual por empresa

In [9]:
# ── 7. Evolución temporal mensual Reddit ──────────────────────────────────
reddit['mes'] = reddit['fecha'].dt.to_period('M')
reddit_mensual = reddit.groupby(['mes', 'ticker'])['n_posts'].sum().reset_index()
reddit_mensual['mes_dt'] = reddit_mensual['mes'].dt.to_timestamp()

fig, axes = plt.subplots(4, 2, figsize=(16, 14), sharex=True)
axes_flat = axes.flatten()

for i, t in enumerate(TICKERS):
    ax = axes_flat[i]
    sub = reddit_mensual[reddit_mensual['ticker'] == t].sort_values('mes_dt')
    ax.bar(sub['mes_dt'], sub['n_posts'], width=25, color=COLORES[t], alpha=0.75)
    ax.set_title(f"{t} — {EMPRESAS[t]}", fontsize=11, fontweight='bold')
    ax.set_ylabel('Posts/mes')

axes_flat[7].set_visible(False)

fig.suptitle('Evolución mensual del volumen de posts en Reddit — 2019–2026',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
guardar(fig, 'eda_texto_volumen_reddit.png')

  → Guardada: docs/figuras/eda_texto_volumen_reddit.png


---
## 8. Reddit — Distribución por subreddit

In [10]:
# ── 8. Distribución por subreddit ─────────────────────────────────────────
sub_counts = posts.groupby(['ticker', 'subreddit']).size().unstack(fill_value=0)
# Reordenar subreddits por volumen total
sub_order = sub_counts.sum().sort_values(ascending=False).index.tolist()
sub_counts = sub_counts[sub_order]
sub_counts = sub_counts.reindex(TICKERS)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 8a. Barras horizontales apiladas por ticker
sub_counts.plot(kind='barh', stacked=True, ax=axes[0],
                colormap='Set2', edgecolor='white', lw=0.3)
axes[0].set_xlabel('Número de posts')
axes[0].set_title('Posts por subreddit y empresa', fontsize=12, fontweight='bold')
axes[0].legend(title='Subreddit', fontsize=9, loc='lower right')

# 8b. Porcentaje general por subreddit (pie)
total_sub = posts['subreddit'].value_counts()
colors_pie = plt.cm.Set2(np.linspace(0, 1, len(total_sub)))
axes[1].pie(total_sub.values, labels=total_sub.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Distribución global por subreddit', fontsize=12, fontweight='bold')

fig.suptitle('Reddit: distribución de posts por subreddit — 2019–2026',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
guardar(fig, 'eda_texto_reddit_subreddits.png')

  → Guardada: docs/figuras/eda_texto_reddit_subreddits.png


---
## 9. Word Clouds por empresa (Reddit)

In [11]:
# ── 9. Word Clouds por empresa ────────────────────────────────────────────
# Stopwords personalizadas
CUSTOM_STOPS = STOPWORDS.copy()
CUSTOM_STOPS.update([
    'stock', 'stocks', 'market', 'price', 'buy', 'sell', 'hold',
    'share', 'shares', 'company', 'will', 'like', 'just', 'think',
    'know', 'get', 'going', 'would', 'one', 'even', 'still',
    'also', 'much', 'really', 'right', 'got', 'thing', 'people',
    'good', 'make', 'way', 'time', 'year', 'years', 'day', 'week',
    'lot', 'money', 'dont', 'need', 'want', 'said', 'say', 'see',
    'go', 'come', 'back', 'take', 'look', 'put', 'made', 'let',
    'well', 'new', 'now', 'use', 'first', 'long', 'big', 'high',
    'low', 'last', 'next', 'every', 'many', 'two', 'since',
    # Tickers y nombres de empresas
    'aapl', 'amzn', 'googl', 'meta', 'msft', 'nvda', 'tsla',
    'apple', 'amazon', 'alphabet', 'google', 'microsoft', 'nvidia',
    'tesla', 'facebook', 'meta platforms',
    'AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA',
])

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes_flat = axes.flatten()

for i, t in enumerate(TICKERS):
    ax = axes_flat[i]
    textos = posts.loc[posts['ticker'] == t, 'texto_completo'].dropna()
    corpus = ' '.join(textos.astype(str).tolist())

    if len(corpus.strip()) < 50:
        ax.text(0.5, 0.5, 'Sin texto suficiente', ha='center', va='center',
                fontsize=12, transform=ax.transAxes)
    else:
        wc = WordCloud(width=800, height=400, max_words=120,
                       background_color='white', colormap='viridis',
                       stopwords=CUSTOM_STOPS, collocations=False,
                       random_state=42).generate(corpus)
        ax.imshow(wc, interpolation='bilinear')

    ax.set_title(f"{t} — {EMPRESAS[t]}", fontsize=12, fontweight='bold')
    ax.axis('off')

    # Guardar también por separado
    fig_ind, ax_ind = plt.subplots(figsize=(10, 5))
    if len(corpus.strip()) >= 50:
        ax_ind.imshow(wc, interpolation='bilinear')
    ax_ind.axis('off')
    ax_ind.set_title(f"Word Cloud Reddit — {t} ({EMPRESAS[t]}) — 2019–2026",
                     fontsize=13, fontweight='bold')
    fig_ind.tight_layout()
    guardar(fig_ind, f'reddit_wordcloud_{t}.png')

# Ocultar subplot sobrante
axes_flat[7].set_visible(False)

fig.suptitle('Word Clouds de posts de Reddit por empresa — 2019–2026',
             fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
guardar(fig, 'reddit_wordclouds_todos.png')

  → Guardada: docs/figuras/reddit_wordcloud_AAPL.png


  → Guardada: docs/figuras/reddit_wordcloud_AMZN.png


  → Guardada: docs/figuras/reddit_wordcloud_GOOGL.png


  → Guardada: docs/figuras/reddit_wordcloud_META.png


  → Guardada: docs/figuras/reddit_wordcloud_MSFT.png


  → Guardada: docs/figuras/reddit_wordcloud_NVDA.png


  → Guardada: docs/figuras/reddit_wordcloud_TSLA.png


  → Guardada: docs/figuras/reddit_wordclouds_todos.png


---
## 10. Correlación cruzada: Tono GDELT vs Volumen Reddit

In [12]:
# ── 10. Correlación cruzada: tono GDELT vs volumen Reddit ─────────────────
# Merge ambos datasets por fecha y ticker
merged = news[['fecha', 'ticker', 'tone_mean', 'n_noticias']].merge(
    reddit[['fecha', 'ticker', 'n_posts', 'score_mean']],
    on=['fecha', 'ticker'], how='inner'
)

print(f"Observaciones con ambas fuentes: {len(merged):,}")
print(f"Periodo solapado: {merged['fecha'].min().date()} → {merged['fecha'].max().date()}")
print()

# 10a. Scatter: tono GDELT vs n_posts Reddit (por empresa)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes_flat = axes.flatten()

for i, t in enumerate(TICKERS):
    ax = axes_flat[i]
    sub = merged[merged['ticker'] == t]
    ax.scatter(sub['tone_mean'], sub['n_posts'], alpha=0.2, s=8,
               color=COLORES[t], edgecolors='none')
    # Correlación
    corr = sub[['tone_mean', 'n_posts']].corr().iloc[0, 1]
    ax.set_title(f"{t} (r={corr:.3f})", fontsize=11, fontweight='bold')
    ax.set_xlabel('Tono medio GDELT')
    ax.set_ylabel('Posts Reddit/día')

axes_flat[7].set_visible(False)

fig.suptitle('Tono GDELT vs volumen de posts Reddit — 2019–2026',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
guardar(fig, 'eda_texto_scatter_tono_reddit.png')

Observaciones con ambas fuentes: 14,929
Periodo solapado: 2019-01-01 → 2026-03-31



  → Guardada: docs/figuras/eda_texto_scatter_tono_reddit.png


In [13]:
# ── 10b. Heatmap de correlaciones cruzadas (tono, volumen noticias, posts, score)
corr_vars = ['tone_mean', 'n_noticias', 'n_posts', 'score_mean']
corr_labels = ['Tono GDELT', 'N. noticias', 'Posts Reddit', 'Score Reddit']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes_flat = axes.flatten()

for i, t in enumerate(TICKERS):
    ax = axes_flat[i]
    sub = merged[merged['ticker'] == t][corr_vars].dropna()
    corr_mat = sub.corr()
    sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                xticklabels=corr_labels, yticklabels=corr_labels,
                square=True, ax=ax, cbar=False, linewidths=0.5,
                annot_kws={'size': 8})
    ax.set_title(t, fontsize=11, fontweight='bold')
    ax.tick_params(labelsize=8)

axes_flat[7].set_visible(False)

fig.suptitle('Correlación cruzada: GDELT vs Reddit por empresa — 2019–2026',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
guardar(fig, 'eda_texto_cross_correlacion.png')

  → Guardada: docs/figuras/eda_texto_cross_correlacion.png


---
## 11. Resumen cuantitativo

In [14]:
# ── 11. Resumen cuantitativo ──────────────────────────────────────────────
print("=" * 70)
print("RESUMEN CUANTITATIVO — EDA TEXTO")
print("=" * 70)
print()
print("GDELT (noticias)")
print(f"  Periodo: {news['fecha'].min().date()} → {news['fecha'].max().date()}")
print(f"  Observaciones (ticker-día): {len(news):,}")
print(f"  Total noticias procesadas: {news['n_noticias'].sum():,.0f}")
print(f"  Tono medio global: {news['tone_mean'].mean():.4f}")
print(f"  % días con tono negativo: {(news['tone_mean'] < 0).mean()*100:.1f}%")
print(f"  Empresa con tono más negativo: {news.groupby('ticker')['tone_mean'].mean().idxmin()} "
      f"({news.groupby('ticker')['tone_mean'].mean().min():.4f})")
print(f"  Empresa con tono más positivo: {news.groupby('ticker')['tone_mean'].mean().idxmax()} "
      f"({news.groupby('ticker')['tone_mean'].mean().max():.4f})")
print()
print("Reddit")
print(f"  Periodo: {reddit['fecha'].min().date()} → {reddit['fecha'].max().date()}")
print(f"  Observaciones (ticker-día): {len(reddit):,}")
print(f"  Total posts individuales: {len(posts):,}")
print(f"  Posts/día medio global: {reddit['n_posts'].mean():.1f}")
print(f"  Subreddits: {sorted(posts['subreddit'].dropna().unique())}")
print(f"  Empresa con más posts: {posts.groupby('ticker').size().idxmax()} "
      f"({posts.groupby('ticker').size().max():,})")
print(f"  Empresa con menos posts: {posts.groupby('ticker').size().idxmin()} "
      f"({posts.groupby('ticker').size().min():,})")
print()
print("Solapamiento (ambas fuentes)")
print(f"  Observaciones con GDELT + Reddit: {len(merged):,}")
print(f"  Periodo solapado: {merged['fecha'].min().date()} → {merged['fecha'].max().date()}")
print()
print("Figuras generadas en docs/figuras/:")
print("  - eda_texto_tono_distribucion.png")
print("  - eda_texto_tono_temporal.png")
print("  - eda_texto_volumen_noticias.png")
print("  - eda_texto_correlaciones_heatmap.png")
print("  - eda_texto_volumen_reddit.png")
print("  - eda_texto_reddit_subreddits.png")
print("  - reddit_wordcloud_{TICKER}.png (x7)")
print("  - reddit_wordclouds_todos.png")
print("  - eda_texto_scatter_tono_reddit.png")
print("  - eda_texto_cross_correlacion.png")
print("=" * 70)

RESUMEN CUANTITATIVO — EDA TEXTO

GDELT (noticias)
  Periodo: 2019-01-01 → 2026-03-31
  Observaciones (ticker-día): 18,409
  Total noticias procesadas: 6,745,505
  Tono medio global: 0.1485
  % días con tono negativo: 37.1%
  Empresa con tono más negativo: GOOGL (-0.2454)
  Empresa con tono más positivo: NVDA (0.6317)

Reddit
  Periodo: 2019-01-01 → 2026-04-02
  Observaciones (ticker-día): 15,047
  Total posts individuales: 99,911
  Posts/día medio global: 6.6
  Subreddits: ['StockMarket', 'investing', 'options', 'stocks', 'wallstreetbets']
  Empresa con más posts: TSLA (34,062)
  Empresa con menos posts: GOOGL (3,595)

Solapamiento (ambas fuentes)
  Observaciones con GDELT + Reddit: 14,929
  Periodo solapado: 2019-01-01 → 2026-03-31

Figuras generadas en docs/figuras/:
  - eda_texto_tono_distribucion.png
  - eda_texto_tono_temporal.png
  - eda_texto_volumen_noticias.png
  - eda_texto_correlaciones_heatmap.png
  - eda_texto_volumen_reddit.png
  - eda_texto_reddit_subreddits.png
  - red